# Error analysis

Where does the remaining out-of-fold error come from? This analysis was repeated after each larger development step to
decide where to work next; the version here uses the final corrector run. The error is squared, so a few large
misses can dominate a month.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir() and (p / "pyproject.toml").is_file())
FEATURES = ROOT / "data/03_features"
sys.path.insert(0, str(ROOT / "model"))
import runner

runner.configure_tracking()
CORRECTOR_RUN = "295"   # run of 09_corrector

In [2]:
oof = pd.read_parquet(runner.artifact_dir(runner.resolve_run(CORRECTOR_RUN)) / "oof_predictions.parquet")
delay = pd.read_parquet(FEATURES / "train_DEP_merged_sorted.parquet", columns=["delay_sec"])["delay_sec"]
oof["delay_sec"] = delay.to_numpy()[oof["row_id"]]
same = (oof["target"] - oof["delay_sec"]).abs() <= 60
oof["regime"] = np.select([same & (oof["delay_sec"] > 2000), same, oof["target"] > 7200],
                          ["artefact", "on-time pushback", "long taxi"], "normal")
oof["sq_error"] = (oof["final"] - oof["target"]) ** 2
oof.groupby("month")["sq_error"].mean().pow(.5).round(1).to_frame("rmse")

,rmse
month,
1,328.0
2,218.9
3,181.5
4,182.5
5,275.0
6,294.3
7,271.0
8,223.7
9,229.8


### Share of the squared error

By label regime, by flights with and without NM off-block, and by airport (percent of each month's squared error).

In [3]:
def share(by):
    g = oof.groupby(["month", by])["sq_error"].sum()
    return (g / g.groupby(level=0).transform("sum") * 100).unstack().round(1)

share("regime")

regime,artefact,long taxi,normal,on-time pushback
month,,,,
1,0.2,64.1,34.4,1.3
2,0.3,9.9,87.1,2.7
3,0.3,3.4,92.6,3.7
4,0.2,1.7,93.3,4.8
5,0.1,58.3,39.9,1.7
6,0.2,48.7,49.6,1.5
7,0.5,16.1,81.2,2.2
8,0.4,6.7,90.1,2.8
9,0.2,20.5,76.9,2.4


In [4]:
share("lane")

lane,NM off-block,no NM off-block
month,,
1,33.0,67.0
2,79.3,20.7
3,88.2,11.8
4,90.2,9.8
5,40.2,59.8
6,43.0,57.0
7,67.5,32.5
8,79.0,21.0
9,67.9,32.1


In [5]:
share("ADEP_mvt")

ADEP_mvt,EDDF,EDDM,EGLL,EHAM,LEBL,LEMD,LFPG,LIRF,LSZH,LTFM
month,,,,,,,,,,
1,2.2,1.1,5.6,3.5,1.0,2.6,69.1,7.3,1.7,5.9
2,3.2,1.9,9.2,3.8,2.0,4.9,9.9,18.8,6.7,39.5
3,4.4,2.2,16.8,4.9,3.9,7.2,14.0,26.0,2.5,18.2
4,4.5,2.2,12.1,7.0,4.3,4.9,13.0,30.2,2.3,19.4
5,2.8,1.2,5.7,1.8,2.3,2.4,6.9,14.1,55.1,7.7
6,2.4,1.7,6.1,1.7,2.4,2.2,7.6,67.2,2.0,6.6
7,3.8,2.9,11.5,2.8,7.1,4.7,9.0,43.2,2.3,12.7
8,3.9,2.5,9.8,3.0,8.6,6.1,12.5,38.2,2.3,13.0
9,3.5,3.7,9.9,3.2,7.3,4.4,11.0,44.8,1.7,10.6


### Largest misses

The 20 largest errors per month and how much of the month's squared error they carry.

In [6]:
top = oof.groupby("month")["sq_error"].apply(lambda e: e.nlargest(20).sum() / e.sum() * 100).round(1)
top.to_frame("top 20 share (%)")

,top 20 share (%)
month,
1,65.7
2,11.9
3,9.4
4,9.4
5,59.6
6,52.4
7,18.9
8,15.9
9,26.8


In [7]:
cols = ["month", "ADEP_mvt", "lane", "regime", "target", "delay_sec", "base_prediction", "p_artifact", "final"]
oof.nlargest(15, "sq_error")[cols].round(2)

,month,ADEP_mvt,lane,regime,target,delay_sec,base_prediction,p_artifact,final
718145,5,LSZH,no NM off-block,long taxi,87341,1438.0,406.21,0.00,496.73
974946,6,LIRF,no NM off-block,long taxi,87168,3485.0,5061.56,0.15,2133.72
93555,1,LFPG,no NM off-block,long taxi,84240,1315.0,641.95,0.00,625.24
112160,1,LFPG,no NM off-block,long taxi,58206,1965.0,982.61,0.00,1007.31
1554680,9,LIRF,no NM off-block,long taxi,87543,692.0,60600.42,0.86,52064.51
1183677,7,LIRF,no NM off-block,long taxi,87186,1019.0,61931.27,0.20,61495.41
644354,5,LIRF,no NM off-block,long taxi,31268,1617.0,7220.09,0.00,8768.44
1366195,8,LIRF,no NM off-block,normal,1078,1257.0,20475.02,0.24,23320.71
1539236,9,LIRF,no NM off-block,long taxi,64849,1489.0,69222.51,0.42,84965.56
1084291,7,LIRF,no NM off-block,long taxi,64437,3455.0,65831.20,0.72,83738.00


### What follows

In January, May and June most of the squared error comes from long-taxi labels of flights without NM off-block, each
month at one airport (LFPG in January, LSZH in May, LIRF in June), and the 20 largest misses carry more than half of it.
The very largest are labels of about one day. Most long-taxi labels are a few hours, and the models predict most of them
poorly. The one pattern found is at LIRF, for flights without NM off-block and a delay of 17–26 hours, which the rule in
`09_corrector` handles. In the other months the error is spread over normal flights, which is where the base models and
the corrector work.